# Analyzing epigenomics result for D-gene identification attempt

In [3]:
import pandas as pd

In [4]:
# Load the annotated DMR table (tab-separated).
# One row per (DMR x feature x gene); contains all 6 pairwise sample comparisons.
df = pd.read_csv(
    "/home/daffa/Work/2026/thesis/results/DMR/DMR_annotated.tsv",
    sep="\t",
)
df.head()

,chr,start,end,length,nCG,meanMethy1,meanMethy2,diff.Methy,areaStat,sample_a,sample_b,direction,feature,gene_label
0,NC_012873.2,67594394,67601630,7237,3447,0.244203,0.056058,0.188145,19719.767762,SBC10,SBC4,hyper_SBC10,promoter,LOC110435067
1,NC_012870.2,72402810,72407599,4790,2449,0.218997,0.045744,0.173253,15737.519672,SBC10,SBC4,hyper_SBC10,intergenic,NaN
2,NC_012870.2,8087755,8092275,4521,2413,0.265127,0.096703,0.168424,13173.727692,SBC10,SBC4,hyper_SBC10,intergenic,NaN
3,NC_012873.2,10883151,10887512,4362,2303,0.256243,0.080646,0.175597,13112.356467,SBC10,SBC4,hyper_SBC10,promoter,LOC8078332
4,NC_012870.2,46238158,46242655,4498,2064,0.225253,0.052197,0.173056,13096.312671,SBC10,SBC4,hyper_SBC10,intergenic,NaN


## Filter: DMRs hypermethylated in the juicy (high-juice) samples vs the dry sample

**Hypothesis:** the D-gene (a Dry-stalk transcription factor) is silenced by promoter methylation in juicy stems, so the **high-juice** samples should carry promoter hypermethylation that the **low-juice** sample lacks.

Juice production (CLAUDE.md): SBC10 `+++`, SBC4 `++`, SBC23 `++` → **high juice**; SBC11 `-` → **low juice (dry)**.

So the contrast is the three high-juice samples vs **SBC11**, using the three SBC11-anchored pairs. A row is "hyper in a high-juice sample" when it involves SBC11 and `direction != hyper_SBC11`. We compute `delta_vs_SBC11 = methylation(high-juice) − methylation(SBC11)` (positive = methylation gained in the juicy sample); the sign of `diff.Methy` depends on which sample is group1, so we recompute from `meanMethy1/2`. The strongest D-gene evidence is a **promoter hypermethylated in all three** high-juice samples.

In [5]:
# --- Core filter: DMRs hypermethylated in a high-juice sample vs the dry SBC11 ---
LOW_JUICE  = "SBC11"                       # dry stem, juice production "-"
HIGH_JUICE = ["SBC10", "SBC4", "SBC23"]    # juicy stems (+++, ++, ++)

# Keep the 3 SBC11-anchored pairs, hypermethylated in the non-SBC11 (high-juice) member.
mask  = ((df["sample_a"] == LOW_JUICE) | (df["sample_b"] == LOW_JUICE)) \
        & (df["direction"] != f"hyper_{LOW_JUICE}")
hyper = df[mask].copy()
hyper["high_sample"]    = hyper["direction"].str.replace("hyper_", "", regex=False)
# meanMethy1 = sample_a, meanMethy2 = sample_b; pick whichever side is the high-juice sample.
hyper["meth_high"]      = hyper["meanMethy2"].where(hyper["sample_a"] == LOW_JUICE, hyper["meanMethy1"])
hyper["meth_SBC11"]     = hyper["meanMethy1"].where(hyper["sample_a"] == LOW_JUICE, hyper["meanMethy2"])
hyper["delta_vs_SBC11"] = hyper["meth_high"] - hyper["meth_SBC11"]  # + = methylation gained in high-juice

print(f"{len(hyper):,} DMR rows hypermethylated in a high-juice sample (relative to SBC11)")
print(hyper["high_sample"].value_counts())
print(hyper["feature"].value_counts())
hyper.sort_values("delta_vs_SBC11", ascending=False).head(20)

46,313 DMR rows hypermethylated in a high-juice sample (relative to SBC11)
high_sample
SBC23    19683
SBC10    16188
SBC4     10442
Name: count, dtype: int64
feature
intergenic    35302
promoter       6961
exon           2251
CDS            1799
Name: count, dtype: int64


,chr,start,end,length,nCG,meanMethy1,meanMethy2,diff.Methy,areaStat,sample_a,sample_b,direction,feature,gene_label,high_sample,meth_high,meth_SBC11,delta_vs_SBC11
79191,NC_012875.2,49695974,49696411,438,67,0.022883,0.769896,-0.747013,-1676.414621,SBC11,SBC4,hyper_SBC4,intergenic,NaN,SBC4,0.769896,0.022883,0.747013
98830,NC_012878.2,50181341,50181668,328,46,0.003419,0.736224,-0.732805,-1163.523727,SBC11,SBC23,hyper_SBC23,intergenic,NaN,SBC23,0.736224,0.003419,0.732805
108015,NC_012873.2,9781452,9781721,270,28,0.051447,0.777984,-0.726537,-491.385624,SBC11,SBC23,hyper_SBC23,CDS,LOC110434730,SBC23,0.777984,0.051447,0.726537
108014,NC_012873.2,9781452,9781721,270,28,0.051447,0.777984,-0.726537,-491.385624,SBC11,SBC23,hyper_SBC23,exon,LOC110434730,SBC23,0.777984,0.051447,0.726537
94043,NC_012871.2,74215896,74216987,1092,115,0.066098,0.769341,-0.703243,-2256.553054,SBC11,SBC23,hyper_SBC23,intergenic,NaN,SBC23,0.769341,0.066098,0.703243
85230,NC_012879.2,26185391,26185649,259,33,0.050879,0.718248,-0.667369,-565.110694,SBC11,SBC4,hyper_SBC4,intergenic,NaN,SBC4,0.718248,0.050879,0.667369
31499,NC_012876.2,12479058,12479306,249,29,0.679402,0.015199,0.664202,510.156772,SBC10,SBC11,hyper_SBC10,intergenic,NaN,SBC10,0.679402,0.015199,0.664202
110221,NC_012873.2,84222,84525,304,28,0.057438,0.717153,-0.659715,-392.986506,SBC11,SBC23,hyper_SBC23,intergenic,NaN,SBC23,0.717153,0.057438,0.659715
104421,NC_012875.2,32141828,32142375,548,48,0.050457,0.708857,-0.658400,-689.928979,SBC11,SBC23,hyper_SBC23,intergenic,NaN,SBC23,0.708857,0.050457,0.658400
104757,NC_012874.2,10447395,10448159,765,46,0.027727,0.683503,-0.655776,-668.746100,SBC11,SBC23,hyper_SBC23,intergenic,NaN,SBC23,0.683503,0.027727,0.655776


In [6]:
# Promoter-focused: the hypothesis is specifically about promoter methylation.
prom = hyper[hyper["feature"] == "promoter"].copy()

# Per gene, the strongest promoter hypermethylation (max delta) in each high-juice sample.
pivot = (
    prom.pivot_table(index="gene_label", columns="high_sample",
                     values="delta_vs_SBC11", aggfunc="max")
        .reindex(columns=HIGH_JUICE)
)
pivot["n_high_juice"] = pivot.notna().sum(axis=1)

# D-gene candidates: promoter hypermethylated in ALL THREE high-juice samples vs SBC11.
candidates = pivot[pivot["n_high_juice"] == 3].copy()
candidates["min_delta"] = candidates[HIGH_JUICE].min(axis=1)   # rank by weakest of the 3
candidates = candidates.sort_values("min_delta", ascending=False)

print(f"{len(candidates)} genes with promoter hypermethylated in all 3 high-juice samples")
candidates

123 genes with promoter hypermethylated in all 3 high-juice samples


high_sample,SBC10,SBC4,SBC23,n_high_juice,min_delta
gene_label,,,,,
LOC8078033,0.289771,0.291515,0.282116,3,0.282116
LOC8069001,0.293840,0.273824,0.260232,3,0.260232
LOC8064596,0.232437,0.201121,0.270194,3,0.201121
LOC110429757,0.185025,0.182946,0.184497,3,0.182946
LOC8059829,0.168812,0.175018,0.182967,3,0.168812
...,...,...,...,...,...
LOC8073710,0.116541,0.125889,0.473530,3,0.116541
LOC8056158,0.123725,0.115551,0.138278,3,0.115551
LOC110437166,0.130366,0.195395,0.113293,3,0.113293


In [11]:
# Save the filtered lists
out_dir = "/home/daffa/Work/2026/thesis/analysis/DMR"
hyper.to_csv(f"{out_dir}/DMR_hyper_highjuice.tsv", sep="\t", index=False)
prom.to_csv(f"{out_dir}/DMR_hyper_highjuice_promoter.tsv", sep="\t", index=False)
candidates.to_csv(f"{out_dir}/dgene_promoter_candidates.tsv", sep="\t")

print("Wrote to", out_dir)
print(" - DMR_hyper_highjuice.tsv          (all hyper-in-high-juice DMRs, every feature)")
print(" - DMR_hyper_highjuice_promoter.tsv (promoter subset)")
print(" - dgene_promoter_candidates.tsv    (genes hyper in all 3 high-juice samples)")

Wrote to /home/daffa/Work/2026/thesis/analysis/DMR
 - DMR_hyper_highjuice.tsv          (all hyper-in-high-juice DMRs, every feature)
 - DMR_hyper_highjuice_promoter.tsv (promoter subset)
 - dgene_promoter_candidates.tsv    (genes hyper in all 3 high-juice samples)
